# Does Γ's new gating tell a learned update from one that learned nothing?

*E46 stage 0, `experiments/regating_table/`, 2026-10-29. Record:
`reports/analyses/2026-10-29_analysis_e46-regating-table.md`. Plan: `reports/plans/2026-10-28_plan_regating-line-reset.md`.
Red-team review: `reports/reviews/2026-10-28_review_regating-line-red-team.md`.*

## The answer

Mostly yes, on toy tasks, with one check left open. When the importance function is fed the fine-tuned model's
activations, the stand-in model's prediction of the answer rises by several units for every real fine-tune. It stays near
zero or falls for every update that learned nothing or learned something else. A hand-made write that only points the
model at the answer earns well under the copy fine-tunes, so the test meant to retire this line did not fire. But that write
lands close to the fine-tunes that teach a new mapping. The random-noise check was too weak on six tasks of seven to settle
whether Γ reacts to size alone. The next step is the user's call.

## What we were trying to test

The user asked whether the change in Γ's values after a fine-tune "is genuinely describing how the model has changed, or
just some strange after-effect of it trying to predict a model it isnt trained on". A red-team review then showed that the
old headline share reads about the same for an update that learned nothing. Part of the reason is arithmetic: the share
divides by the real gain, which is near zero for such an update. So this stage drops the share and reads one number, in
nats, on one table, next to its nulls.

**The number.** Γ's values are computed twice on the same inputs, from the original model's activations and from the
fine-tuned model's. The **gate gain** is $g = L(C_0, G_1) - L(C_0, G_0)$:

- $L$, the CI-masked model's mean log-odds of the target at the scored position;
- $C_0$, the original subcomponents, so the update itself is left out;
- $G_0$ and $G_1$, Γ's values from the original and from the fine-tuned model.

A gate gain of 0 means Γ's new values give the old subcomponents nothing. The **real gain** $G = T_1 - T_0$ is the same
log-odds change in the full model, the yardstick. The gate gain lives in the masked stand-in only; it is never read in
the real fine-tuned model.

**The updates.** Seven toy fine-tunes of the 67M model, each a few Adam steps at two seeds, plus one that learned nothing.

- Three single-trigger tasks: after ` :)` predict `o`; after ` which` predict ` was`; after ` we` predict ` just`.
- Two copy tasks: after a marker and a key word, repeat the key, 32 keys per table. The model already copies, so this
  strengthens existing behaviour by construction.
- Two random-map tasks: the same keys, each mapped to a fixed random word. Nothing in the model does this, so it is new
  behaviour by construction.
- A fine-tune on code text that learned nothing: held-out text got 0.05 nats per token worse.

**The nulls.** Each is a forward pass on a changed copy of the original weights. Nothing is trained.

- *A fine-tune to another word*, ` http` after the same trigger: does any learned update of this size raise the target?
- *The fine-tune's update with its sign flipped*: does Γ follow the direction of the change?
- *A random update matched on side effects*: Gaussian, scaled until the full model's KL to the original on positions the
  task does not score equals the fine-tune's, capped at ten times the fine-tune's norm. Does Γ react to a disturbance that
  points nowhere in particular?
- *A per-key write*: one change to block 1's MLP output matrix that maps each key's mean input onto its answer's output
  direction, scaled until its real gain matches the fine-tune's. It uses no existing machinery. Does Γ hand the old
  subcomponents a gain whenever the activations start to predict the target?

The write is scored only on the random maps. On the copy tasks the answer is the key itself, and the model ties its input and
output embeddings, so the write there just adds the key's own embedding: an existing representation.

**The floor.** Per state, the larger of 0.1 nats and twice the standard error of the gate gain over held-out windows.

## What we expected, and why

- **The copy and random-map fine-tunes reproduce the earlier experiment.** Take what the updated subcomponents do
  unmasked. Under Γ's new values copy should keep at least 0.8 of it, and the random maps at most 0.5. This checks the
  rebuild; it is not new evidence.
- **The random update stays inside the floor.** An earlier size-matched random update was inert on emoticon. Matching side
  effects instead of size was meant to give it a fair chance. A quarter of the fine-tune's gate gain on any task would
  fail it; between the floor and that bar is inconclusive.
- **Nothing that lowers the target raises the gate gain.** The other-target fine-tunes and flipped updates should read at
  or below the floor.
- **The write reads well below the copy fine-tune.** If Γ simply turns on the answer's machinery whenever the activations
  predict the answer, the write would read like the copy fine-tune. At 0.8 of it or more the line is retired; at 0.5 or
  less it passes. We leaned medium.

## First, the control

The first control run stopped at once. The ` http` fine-tune lowered the emoticon target by about ten nats, and the filed
rule had asked for its real gain to stay near zero on both sides. That rule could never pass: learning ` http` after
` :)` must take probability from `o`. The user made the rule one-sided, so the null passes when it does not *raise* the
target. The rerun passed every check. The original model's readings matched the earlier experiments to within a hundredth
of a nat. The saved and rebuilt fine-tunes matched their filed gains within the 2% tolerance.

In [1]:
# Stage 0a, the control, from the committed results (no model).
import json
from pathlib import Path

R = Path("../experiments/regating_table/results")
a = json.loads((R / "stage0a.json").read_text())
print(f"{'group':22s} {'passed':>7s}")
for name, grp in a["groups"].items():
    print(f"{name:22s} {str(grp['ok']):>7s}")
print("whole control passed:", a["pass"])

group                   passed
e12/build                 True
e12/ident32               True
e12/ident32b              True
e12/rand32                True
e12/rand32b               True
e18/emoticon              True
e18/r1_t534_v369          True
e18/r2_t359_v816          True
e36/code                  True
whole control passed: True


## Does Γ's new gating help the old subcomponents for the fine-tunes but not for the nulls?

![gate gain of each fine-tune against each null on the same task](../experiments/regating_table/figures/gate_gain.png)

**Reading it.** Yes, with one weak spot. Every fine-tune, the blue circles, gives the old subcomponents 3.3 to 11.6 nats
through Γ alone. Every update that lowers the target reads a negative gate gain. Those are the other-target fine-tunes in
purple and the flipped updates in orange. The random updates, grey squares, sit within 0.16 nats of zero on all seven
tasks. Five are inside the shaded floor. Two are just outside it: emoticon reads −0.155 and random map A +0.118. Each is
far below its red fail bar, which sits at 2.0 and 0.94 nats. The green triangles are the per-key writes, which land
between the nulls and the fine-tunes. The axis is stretched near zero, so small values look larger than they are.

In [2]:
# Gate gain per task in nats: seed-0 fine-tune and each null.
b = json.loads((R / "stage0b.json").read_text())
S = b["states"]
TASKS = {"emoticon": "emoticon", "r1_t534_v369": "which", "r2_t359_v816": "we", "ident32": "copy A",
         "ident32b": "copy B", "rand32": "random map A", "rand32b": "random map B"}


def g(key):
    return f"{S[key]['g']:8.2f}" if key in S else f"{'':>8s}"


print(f"{'task':13s} {'fine-tune':>9s} {'random':>8s} {'flipped':>8s} {'write':>8s}")
for t, name in TASKS.items():
    print(f"{name:13s} {S[t + '/s0']['g']:9.2f} {g(t + '/random')} {g(t + '/flipped')} {g(t + '/write')}")

task          fine-tune   random  flipped    write
emoticon           8.02    -0.16    -2.76         
which              7.69    -0.08    -4.82         
we                 3.56    -0.08    -1.32         
copy A            11.23     0.08    -1.49     5.17
copy B            11.57     0.04    -1.68     6.71
random map A       3.75     0.12    -1.41     1.80
random map B       6.01     0.05    -0.86     3.92


## Did each random update reach the fine-tune's side effects?

![KL each random update reached against the fine-tune's](../experiments/regating_table/figures/random_kl.png)

**Reading it.** Only on one task of seven. Random map B, the filled point, matched the fine-tune's off-target KL at 6.5
times its norm. The other six hit the cap at ten times and stopped short. Emoticon got closest, at 0.80 of its target.
Random map A got 0.12. So on six tasks the random null disturbed the rest of the text less than the fine-tune did. A
null weaker than its target passes more easily than the rule intended. Over the last grid points the KL grew
about as the square of the scale. On that trend, matching would have needed 11 to 29 times the norm; that is an
extrapolation, not a measurement.

In [3]:
# Off-target KL of each random update against its target, the fine-tune's.
print(f"{'task':13s} {'target':>7s} {'reached':>8s} {'ratio':>6s} {'ended':>8s}")
for t, name in TASKS.items():
    r = S[t + "/random"]
    print(f"{name:13s} {r['kl_target']:7.3f} {r['kl_other']:8.3f} {r['kl_other'] / r['kl_target']:6.2f} {r['match']:>8s}")

task           target  reached  ratio    ended
emoticon        0.055    0.044   0.80      cap
which           0.122    0.055   0.46      cap
we              0.090    0.046   0.51      cap
copy A          0.189    0.070   0.37      cap
copy B          0.205    0.072   0.35      cap
random map A    0.696    0.085   0.12      cap
random map B    0.097    0.107   1.10       ok


## Does Γ treat a write that only points the activations at the answer like the copy fine-tune?

![per-key write's gate gain over the copy fine-tune's, with the pass and retire lines](../experiments/regating_table/figures/write_rho.png)

**Reading it.** No, but Γ does react to it. The scored writes, filled green triangles, read 0.16 and 0.34 of the copy
fine-tune's gate gain. Both are under the pass line at 0.5, so the retire test did not fire. The blue squares give the
proportion that matters: the real random-map fine-tunes read 0.31 to 0.52 of the copy fine-tune. The write sits below
the new-behaviour fine-tune, but not far below it. In nats it gave the old subcomponents 1.8 and 3.9, half and two-thirds of what the
random-map fine-tunes gave them. A write that uses no existing machinery still earns a gate gain far above the floor.
The hollow triangles are the copy writes, which add the key's own embedding. They read 0.46 and 0.58 and are not scored.

In [4]:
# The write test: gate gain over the seed-0 copy fine-tune's, for the same table size.
pairs = {"rand32": "ident32", "rand32b": "ident32b"}
print(f"{'table':6s} {'write':>6s} {'map ft s0':>10s} {'map ft s1':>10s} {'copy write':>11s}")
for rand, copy in pairs.items():
    ref = S[copy + "/s0"]["g"]
    vals = [S[k]["g"] / ref for k in (rand + "/write", rand + "/s0", rand + "/s1", copy + "/write")]
    print(f"{'A' if rand == 'rand32' else 'B':6s} {vals[0]:6.2f} {vals[1]:10.2f} {vals[2]:10.2f} {vals[3]:11.2f}")

table   write  map ft s0  map ft s1  copy write
A        0.16       0.33       0.31        0.46
B        0.34       0.52       0.50        0.58


## Do the copy and random-map fine-tunes reproduce the earlier experiment?

Yes, on all eight states, to the third decimal. Take what the updated subcomponents do unmasked. Under Γ's new values the
copy states keep 0.87 to 0.97 of it, and the random maps keep 0.36 to 0.41. This replicates the earlier result and adds no
new evidence: the gate gain alone already separates the classes, at about 11 nats for copy against 3.5 to 6 for the random maps.

## What we learned

Read in nats rather than as a share, the gate gain rises for updates that raise the target and stays at or below zero for
updates that lower it. That removes the red team's sharpest complaint, which came from dividing by a near-zero gain. It is
not a clean reading of "existing machinery", though. A write that uses no machinery earns half to two-thirds of what a
new-mapping fine-tune earns. `GOALS.md` asks whether post-training strengthens an existing behaviour or teaches a new one.
Here the copy fine-tunes stand apart from both, but the line between "new behaviour" and "no machinery at all" is thin.

## How compelling is the evidence

- The control passed everywhere, and the earlier experiment's numbers were reproduced to the third decimal.
- The random null is the weak link. It matched the fine-tune's side effects on one task only. Its two readings outside the
  floor are 2.5 and 4.6 standard errors from zero, measured across held-out windows, but small against their fail bars.
- Every null was drawn once per task: one random direction, one write, one flipped update. The floor is window noise for
  that one draw, not the spread between draws.
- Everything is on the 67M model with one scored position per task: three single-trigger tasks, two copy tables, two
  random-map tables and one code fine-tune.
- Every gate gain is read in the masked stand-in. Nothing here says what the real fine-tuned model does through these
  subcomponents.

## Surprises

- **The per-key write earns a real gate gain.** It reads 1.8 and 3.9 nats, far above the floor. That is half to two-thirds of
  the random-map fine-tunes. No prediction compared it with them; the filed test used the copy fine-tune only.
- **The random update hit its cap on six tasks of seven.** The cap was filed as a guard, with no expected count.
- The emoticon random update lowers the target a little, by 0.155 nats through Γ. It is outside the floor in size, not in
  the direction that would count against the line.

## What next

The plan's decision tree has no branch for an inconclusive random null beside a passed write test, so it goes to the user
as a decision. Three things bear on it:

- **A stronger random null.** More than one random direction per task, with the scale allowed past ten times until the
  side effects match. If all stay inside the floor, the size question closes. If any reaches a quarter of the fine-tune's
  gate gain, the plan's retire branch applies.
- **Whether the write is really a "no machinery" null.** The cosine between the write's and the fine-tune's change to the
  residual stream, proposed in review and not run. If it is high on the random maps, the write test cannot be read.
- **Stage 1 on the larger model's real updates**, as the plan lays out, only on the user's go-ahead.

## Scorecard

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| 0a | apparatus reproduces earlier runs | cells within 0.02; identity 0.597 ± 0.01; gains within 2%; ` http` null ≤ 0.1 of the gain, one-sided | all within; null −0.72 | pass |
| P0.2 | copy and random maps replicate E12 | gated piece gain over piece gain ≥ 0.8 copy, ≤ 0.5 random, both seeds | 0.87–0.97; 0.36–0.41 | pass |
| P0.3 | KL-matched random update moves no gate gain | all within F; any ≥ 0.25 × fine-tune g fails | 5 of 7 within; −0.155, +0.118 over F = 0.1; cap on 6 of 7 | inconclusive |
| non-positive | G ≤ 0 never raises g | g ≤ 0.1 + F | every such row within | pass |
| P0.4 | random-map write ≠ copy fine-tune | ρ ≤ 0.5 pass, ≥ 0.8 retire | ρ = 0.16, 0.34 | pass |
| tree | | P0.2 pass, P0.3 inconclusive, P0.4 pass | | card for the user |

## Not run

- A second random direction per task; any random scale past ten times the fine-tune's norm.
- A structured null of the fine-tune's size that is not the fine-tune.
- The residual-stream cosine between the write and the fine-tune.
- Writes at other layers or of other forms.
- The lowercase fine-tune, which sits at the floor; the teacher-training states, which no prediction reads.
- Any reading in the real fine-tuned model, any Γ from another seed or refitted, and anything on the larger model.

## Provenance

Control: Slurm 41161 stopped at the null; Slurm 42604 passed, 112 s, `results/stage0a.json`. Table: Slurm 42675, 109 s,
`results/stage0b.json`, log `cluster/logs/commute-regating-table-42675.out`. MATS cluster debug QoS, $0. Figures:
`experiments/regating_table/plot.py`.